# AIAT 125 — Deploying AI Models
## Final Comprehensive Exercise

**Course**: AIAT 125 | **Institution**: Tuwaiq Academy for Training | **Total Points**: 100

---

### Learning Outcomes & Grade Breakdown

| Part | CLO | Topic | Points |
|------|-----|-------|--------|
| Part 1 | CLO1 | Deployment Lifecycle + Model Training | 10 |
| Part 2 | CLO2 | Model Packaging (Pickle/Joblib/ONNX) + Serving Frameworks | 15 |
| Part 3 | CLO3 | REST APIs — FastAPI + Flask | 25 |
| Part 4 | CLO4 | Cloud Deployment (AWS/GCP/Azure) + Security | 10 |
| Part 5 | CLO5 | Docker + Kubernetes (Deployment/Service/HPA) + CI/CD | 15 |
| Part 6 | CLO6 | Monitoring + Alerting + Drift + MLflow/WandB + Versioning + Retraining + A/B + Canary | 25 |
| **Total** | | | **100** |

### Scenario
You are an MLOps engineer at a healthcare company. Your team has trained a **patient risk classification model** and you must deploy it fully to production.

### Instructions:
- Run the **Setup** cell first
- Find `# TODO` and replace `None` / `pass` with the correct code
- Run **ASSERTIONS** cells after each task for immediate verification
- **Final Gate** at the end of the file calculates your total score

In [ ]:
# ── SETUP — Run this cell first ───────────────────────────────────────────────
import subprocess, sys
pkgs = [
    "scikit-learn", "numpy", "pandas", "scipy", "joblib",
    "fastapi", "flask", "httpx", "pydantic", "mlflow",
    "skl2onnx", "onnxruntime"
]
subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + pkgs, check=False)

import os, json, time, pickle, warnings
import numpy as np
import pandas as pd
import joblib
from datetime import datetime
from scipy import stats
from sklearn.datasets import make_classification
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
warnings.filterwarnings('ignore')

BASE_DIR = "/tmp/aiat125_final/"
os.makedirs(BASE_DIR, exist_ok=True)

# Shared dataset: synthetic patient risk data (6 features, binary)
np.random.seed(42)
X_raw, y_raw = make_classification(
    n_samples=1000, n_features=6, n_informative=4, n_redundant=2, random_state=42
)
FEATURE_NAMES = ["age_norm","bp_norm","glucose_norm","bmi_norm","feature_5","feature_6"]
CLASS_NAMES   = ["low_risk", "high_risk"]

X_train, X_test, y_train, y_test = train_test_split(
    X_raw, y_raw, test_size=0.2, random_state=42
)

print(f"Setup complete | Train={len(X_train)} Test={len(X_test)} Features={len(FEATURE_NAMES)}")
print(f"Save dir: {BASE_DIR}")

---
# Part 1 — Deployment Lifecycle (CLO1) — 10 Points

**CLO1**: Understand the lifecycle of deploying AI models.

The six lifecycle stages: **development → testing → packaging → deployment → monitoring → retraining**

---
## Task 1A — Lifecycle Stages Knowledge (5 points)

In [ ]:
# TODO 1A-i: Complete the correct order of stages (1 = first, 6 = last)
LIFECYCLE_STAGES = {
    "development": None,
    "testing":     None,
    "packaging":   None,
    "deployment":  None,
    "monitoring":  None,
    "retraining":  None,
}

# TODO 1A-ii: A brief description for each stage (one sentence)
STAGE_DESCRIPTIONS = {
    "development": None,
    "testing":     None,
    "packaging":   None,
    "deployment":  None,
    "monitoring":  None,
    "retraining":  None,
}

# TODO 1A-iii: What is the feedback loop in deployment? (string)
FEEDBACK_LOOP_DEFINITION = None

for stage, order in sorted(LIFECYCLE_STAGES.items(), key=lambda x: x[1] or 0):
    print(f"  Stage {order}: {stage:15s} — {STAGE_DESCRIPTIONS.get(stage)}")

---
## Task 1B — Train, Validate and Prepare Model (5 points)

In [ ]:
# TODO 1B-i: Train RandomForestClassifier(n_estimators=100, random_state=42)
production_model = None  # YOUR CODE

# TODO 1B-ii: Compute accuracy on X_test
y_pred        = None  # YOUR CODE
test_accuracy = None  # YOUR CODE

# TODO 1B-iii: deployment_ready = True if test_accuracy >= 0.80
DEPLOY_THRESHOLD = 0.80
deployment_ready = None  # YOUR CODE

# TODO 1B-iv: Create deployment_report (dict) with keys:
#   model_type, test_accuracy, threshold, ready, timestamp
deployment_report = None  # YOUR CODE

print(f"Accuracy={test_accuracy:.4f} | Ready={deployment_ready}")

---
# Part 2 — Model Packaging & Serialization (CLO2) — 15 Points

**CLO2**: Package and serialize models for deployment in different formats.

| Format | Use Case |
|--------|----------|
| **Pickle** | Standard Python library, fast |
| **Joblib** | Best for large sklearn models |
| **ONNX** | Open format across different frameworks |
| **TF SavedModel** | TensorFlow deployment format |

---
## Task 2A — Pickle + Joblib Packaging (4 points)

In [ ]:
PICKLE_PATH = os.path.join(BASE_DIR, "model.pkl")
JOBLIB_PATH = os.path.join(BASE_DIR, "model_bundle.joblib")
META_PATH   = os.path.join(BASE_DIR, "model_metadata.json")

# TODO 2A-i: Save production_model with pickle to PICKLE_PATH
pass  # YOUR CODE

# TODO 2A-ii: Create a bundle dict and save it with joblib to JOBLIB_PATH
# Must contain: 'model', 'feature_names', 'class_names', 'metadata'
# metadata contains: 'version', 'accuracy', 'framework', 'trained_at'
model_bundle = None  # YOUR CODE
pass  # YOUR CODE: joblib.dump

# TODO 2A-iii: Save metadata only (without the model) as JSON to META_PATH
pass  # YOUR CODE

# TODO 2A-iv: Reload the joblib bundle
loaded_bundle   = None  # YOUR CODE: joblib.load
loaded_model    = None  # YOUR CODE: loaded_bundle["model"]
loaded_metadata = None  # YOUR CODE: loaded_bundle["metadata"]

if os.path.exists(PICKLE_PATH):
    print(f"Pickle  : {os.path.getsize(PICKLE_PATH):,} bytes")
if os.path.exists(JOBLIB_PATH):
    print(f"Joblib  : {os.path.getsize(JOBLIB_PATH):,} bytes")
print(f"Metadata: {loaded_metadata}")

---
## Task 2B — ONNX Format: Convert, Save, and Run (5 points)

**ONNX** (Open Neural Network Exchange) allows running a model across different frameworks and environments:
- `skl2onnx` — converts scikit-learn models to ONNX
- `onnxruntime` — runs ONNX models with high efficiency

In [ ]:
from skl2onnx import convert_sklearn
from skl2onnx.common.data_types import FloatTensorType
import onnxruntime as rt

ONNX_PATH = os.path.join(BASE_DIR, "model.onnx")

# TODO 2B-i: Convert production_model to ONNX format and save to ONNX_PATH
# Steps:
#   1. initial_type = [("float_input", FloatTensorType([None, 6]))]
#   2. onnx_model = convert_sklearn(production_model, initial_types=initial_type)
#   3. Save: open(ONNX_PATH, "wb").write(onnx_model.SerializeToString())
pass  # YOUR CODE

# TODO 2B-ii: Load the ONNX model using onnxruntime and run inference on X_test[:5]
# Steps:
#   1. sess = rt.InferenceSession(ONNX_PATH)
#   2. input_name = sess.get_inputs()[0].name
#   3. onnx_preds = sess.run(None, {input_name: X_test[:5].astype(np.float32)})[0]
sess        = None  # YOUR CODE
onnx_preds  = None  # YOUR CODE (array of class indices or labels)

# TODO 2B-iii: Answer the following questions
# When should you use ONNX instead of Pickle? (string)
Q_ONNX_WHEN = None
# What is the advantage of onnxruntime over direct sklearn predict()? (string)
Q_ONNX_ADVANTAGE = None

if onnx_preds is not None:
    print(f"ONNX file size : {os.path.getsize(ONNX_PATH):,} bytes")
    print(f"ONNX preds (5) : {onnx_preds[:5]}")
    print(f"sklearn preds  : {production_model.predict(X_test[:5])}")

---
## Task 2C — Serving Frameworks: TF Serving & TorchServe (3 points)

**TensorFlow Serving** and **TorchServe** are production-ready frameworks for serving models:
- Performance optimization (batching, caching)
- Support for multiple model versions
- Built-in monitoring

In [ ]:
# TODO 2C-i: Complete the TF Serving model config as a string
# Must contain: model_config_list, config block, name, base_path, model_platform
TF_SERVING_CONFIG = None  # YOUR CODE — multi-line string

# TODO 2C-ii: Create a TorchServe model archive manifest as a dict
# Must contain the keys:
#   'model_name', 'handler', 'serialized_file', 'model_file', 'version'
TORCHSERVE_MANIFEST = None  # YOUR CODE

# TODO 2C-iii: Answer the questions
# What is the main difference between TF Serving and building a Flask API manually? (string)
Q_TF_VS_FLASK = None
# What is the advantage of multi-model serving support in these frameworks? (string)
Q_MULTI_MODEL = None

# Save files
if TF_SERVING_CONFIG:
    with open(os.path.join(BASE_DIR, "tf_serving.config"), "w") as f:
        f.write(TF_SERVING_CONFIG)
    print(TF_SERVING_CONFIG)
if TORCHSERVE_MANIFEST:
    print(json.dumps(TORCHSERVE_MANIFEST, indent=2))

---
## Task 2D — Batch vs Real-Time Predict Functions (3 points)

In [ ]:
# TODO 2D-i: predict_single(model, features)
# - features: list of 6 numbers
# - Raise ValueError if len(features) != 6 OR any value is not numeric
# - Return: {'prediction': class_name, 'confidence': float(4dp), 'latency_ms': float}
def predict_single(model, features):
    t0 = time.perf_counter()
    pass  # YOUR CODE — validation + predict_proba + return dict


# TODO 2D-ii: predict_batch(model, data)
# - data: 2D array-like, N rows × 6 columns
# - Raise ValueError if data is empty OR any row != 6 columns
# - Return: {'predictions': [class_names], 'confidences': [floats], 'count': int, 'total_time_ms': float}
def predict_batch(model, data):
    t0 = time.perf_counter()
    data = np.array(data)
    pass  # YOUR CODE — validation + batch predict + return dict


print("Single:", predict_single(production_model, X_test[0].tolist()))
print("Batch: ", predict_batch(production_model, X_test[:5]))

---
# Part 3 — Building APIs for Model Serving (CLO3) — 25 Points

**CLO3**: Build REST APIs for serving models.

## Demo — FastAPI Pattern

In [ ]:
from fastapi import FastAPI, HTTPException
from fastapi.testclient import TestClient
from pydantic import BaseModel, Field, validator
from typing import List

# Reference example — do not edit this cell
_demo = FastAPI(title="Demo")
class _DemoReq(BaseModel):
    value: float
@_demo.get("/health")
def _health(): return {"status": "ok"}
@_demo.post("/double")
def _double(req: _DemoReq): return {"result": req.value * 2}
_tc = TestClient(_demo)
print("Health:", _tc.get("/health").json())
print("Double:", _tc.post("/double", json={"value": 5.0}).json())

---
## Task 3A — Complete the Serving App (10 points)

In [ ]:
# TODO 3A-i: Define PredictRequest — Pydantic model containing:
#   - features: List[float] (exactly 6 elements)
#   - validator that raises ValueError if len(features) != 6
class PredictRequest(BaseModel):
    pass  # YOUR CODE

# TODO 3A-ii: Define BatchPredictRequest — containing:
#   - records: List[List[float]]
#   - validator that raises ValueError if records is empty
class BatchPredictRequest(BaseModel):
    pass  # YOUR CODE

# TODO 3A-iii: Create FastAPI app
serving_app = None  # YOUR CODE: FastAPI(title="Patient Risk Classifier", version="1.0.0")

MODEL_REGISTRY = {
    "model":      production_model,
    "metadata":   loaded_metadata,
    "version":    "v1.0.0",
    "loaded_at":  datetime.utcnow().isoformat(),
}

# TODO 3A-iv: GET /health
#   Returns: {"status": "ok", "model_version": ..., "loaded_at": ...}
# YOUR CODE

# TODO 3A-v: POST /predict — accepts PredictRequest
#   Returns: {"prediction": class_name, "confidence": float, "model_version": str}
#   Raises HTTPException(422) if features.length != 6
# YOUR CODE

# TODO 3A-vi: POST /predict/batch — accepts BatchPredictRequest
#   Returns: {"predictions": [...], "confidences": [...], "count": int, "model_version": str}
#   Raises HTTPException(400) if records is empty
# YOUR CODE

if serving_app:
    _c = TestClient(serving_app)
    print(_c.get("/health").json())

---
## Task 3B — Test All Endpoints (5 points)

In [ ]:
_client = TestClient(serving_app) if serving_app else None

# TODO: Test each endpoint and store the results
health_response     = None  # YOUR CODE: _client.get("/health").json()
predict_response    = None  # YOUR CODE: _client.post("/predict", json={"features": X_test[0].tolist()}).json()
invalid_status_code = None  # YOUR CODE: status_code for features with only 3 elements
batch_response      = None  # YOUR CODE: _client.post("/predict/batch", json={"records": X_test[:5].tolist()}).json()
empty_batch_status  = None  # YOUR CODE: status_code for records=[]

print("Health       :", health_response)
print("Predict      :", predict_response)
print("Invalid code :", invalid_status_code)
print("Batch count  :", batch_response.get("count") if batch_response else None)
print("Empty batch  :", empty_batch_status)

---
## Task 3C — Flask API Serving (CLO3) — 10 points

**Flask** is a synchronous WSGI framework — simpler and widely used in legacy codebases.

| | Flask | FastAPI |
|---|---|---|
| Style | Synchronous (WSGI) | Async-first (ASGI) |
| Docs | Manual | Auto OpenAPI/Swagger |
| Validation | Manual | Pydantic (automatic) |
| Production server | Gunicorn | Uvicorn |
| Best for | Simple APIs, legacy systems | High-throughput, data validation |

**Status code rule:** return `400` for bad client input (their fault), `500` for unexpected server errors (your fault).

In [ ]:
import sys as _sys

# TODO 3C-i: Complete the Flask app string below.
# The skeleton is provided — fill in the two TODO sections inside it.
# NOTE: This is a regular string (not f-string), so write normal Python code inside.

FLASK_APP_CODE = """
import joblib, numpy as np
from flask import Flask, request, jsonify

app = Flask(__name__)

MODEL_PATH  = "/tmp/aiat125_final/model_bundle.joblib"
bundle      = joblib.load(MODEL_PATH)
model       = bundle["model"]
CLASS_NAMES = ["low_risk", "high_risk"]

@app.route("/health", methods=["GET"])
def health():
    return jsonify({"status": "ok"}), 200

@app.route("/predict", methods=["POST"])
def predict():
    data = request.get_json(silent=True)

    # TODO 3C-A: Validate input — return 400 for each of these cases:
    #   1. "features" key is missing from data
    #   2. len(data["features"]) != 6
    #   3. any element in data["features"] is not numeric (try float() conversion)
    # Example: return jsonify({"error": "missing features"}), 400
    # YOUR CODE HERE

    # TODO 3C-B: Run inference and return 200 response
    #   features  = np.array([data["features"]])
    #   class_id  = int(model.predict(features)[0])
    #   conf      = round(float(model.predict_proba(features)[0].max()), 3)
    #   return jsonify({"prediction": CLASS_NAMES[class_id], "class_id": class_id, "confidence": conf}), 200
    # YOUR CODE HERE

if __name__ == "__main__":
    app.run(port=5000)
"""

if FLASK_APP_CODE:
    with open("/tmp/flask_serving.py", "w") as _f:
        _f.write(FLASK_APP_CODE)
    if "flask_serving" in _sys.modules:
        del _sys.modules["flask_serving"]
    _sys.path.insert(0, "/tmp")
    import flask_serving as _fm
    flask_client = _fm.app.test_client()
else:
    flask_client = None

# TODO 3C-ii: Test the endpoints and store results
flask_health_status  = None  # YOUR CODE: flask_client.get("/health").status_code
flask_predict_result = None  # YOUR CODE: flask_client.post("/predict", json={"features": X_test[0].tolist()}).get_json()
flask_invalid_status = None  # YOUR CODE: status_code for {"features": [1, 2, 3]}

# TODO 3C-iii: Answer conceptual questions (strings)
Q_FLASK_400_WHEN   = None  # When should you return 400? (client-side vs server-side)
Q_FLASK_500_WHEN   = None  # When should you return 500?
Q_FLASK_GUNICORN   = None  # What is Gunicorn and why use it instead of Flask's dev server?
Q_FLASK_VS_FASTAPI = None  # When would you choose Flask over FastAPI?

print("Flask health :", flask_health_status)
print("Flask predict:", flask_predict_result)
print("Flask invalid:", flask_invalid_status)

---
# Part 4 — Cloud Deployment (CLO4) — 10 Points

**CLO4**: Deploy models on cloud platforms.

---
## Task 4A — Cloud Platforms Knowledge (5 points)

In [ ]:
# TODO 4A-i: Classify each service to its provider ('AWS', 'GCP', or 'Azure')
CLOUD_SERVICES = {
    "SageMaker":       None,
    "Vertex AI":       None,
    "Azure ML":        None,
    "Lambda":          None,
    "Cloud Run":       None,
    "Azure Functions": None,
    "EC2":             None,
    "AKS":             None,  # Azure Kubernetes Service
}

# TODO 4A-ii: Determine the appropriate inference type ('real-time' or 'batch')
INFERENCE_TYPE = {
    "Predict fraud on every credit card transaction as it happens": None,
    "Score 1 million loan applications overnight":                  None,
    "Detect intrusion in a live network stream":                    None,
    "Generate monthly customer churn predictions from CRM data":    None,
    "Return product recommendations while user browses the site":   None,
}

# TODO 4A-iii: Three security best practices for deploying AI on the cloud
CLOUD_SECURITY_PRACTICES = [None, None, None]

print("Cloud services mapping:", CLOUD_SERVICES)

---
## Task 4B — Deployment Config + Authentication + Logging (5 points)

In [ ]:
# TODO 4B-i: cloud deployment config with these required keys:
#   model_name, model_version, endpoint_type (real-time/batch/serverless),
#   instance_type, min_instances, max_instances, autoscaling_enabled (bool),
#   health_check_path, environment_variables (dict), tags (dict: team + project)
cloud_deployment_config = None  # YOUR CODE

# TODO 4B-ii: api_key_auth(provided_key, valid_keys_set) function
#   - True  if provided_key is in valid_keys_set
#   - False if provided_key is None or not found
def api_key_auth(provided_key, valid_keys_set):
    pass  # YOUR CODE

# TODO 4B-iii: log_request(endpoint, status_code, latency_ms) function
#   Returns dict: {'timestamp', 'endpoint', 'status_code', 'latency_ms'}
def log_request(endpoint, status_code, latency_ms):
    pass  # YOUR CODE

VALID_KEYS = {"key-abc-123", "key-xyz-456"}
print(api_key_auth("key-abc-123", VALID_KEYS))
print(api_key_auth("wrong", VALID_KEYS))
print(log_request("/predict", 200, 8.5))

---
# Part 5 — Containers & Orchestration (CLO5) — 15 Points

**CLO5**: Implement containers with Docker and Kubernetes.

---
## Task 5A — Write a Dockerfile (4 points)

In [ ]:
# TODO 5A: Write a Dockerfile for our FastAPI application
# Must contain:
#   FROM python:3.9-slim
#   WORKDIR /app
#   COPY requirements.txt .
#   RUN pip install -r requirements.txt
#   COPY . .
#   EXPOSE 8000
#   CMD ["uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8000"]

DOCKERFILE_CONTENT = None  # YOUR CODE — multi-line string

if DOCKERFILE_CONTENT:
    with open(os.path.join(BASE_DIR, "Dockerfile"), "w") as f:
        f.write(DOCKERFILE_CONTENT)
    print(DOCKERFILE_CONTENT)

---
## Task 5B — Kubernetes: Deployment + Service + HPA (7 points)

Three YAML manifests needed for a complete production ML deployment:

| Manifest | Purpose |
|----------|---------|
| **Deployment** | Runs N replicas of your container; restarts on crash |
| **Service (LoadBalancer)** | Stable external IP that load-balances traffic across all replicas |
| **HPA** | Scales replicas automatically based on CPU utilization |

In [ ]:
# TODO 5B-i: Kubernetes Deployment YAML
# Must contain:
#   apiVersion: apps/v1, kind: Deployment
#   metadata.name: patient-risk-classifier, replicas: 3
#   containerPort: 8000, resources (cpu + memory requests/limits)
#   livenessProbe or readinessProbe
K8S_DEPLOYMENT_YAML = None  # YOUR CODE — multi-line string YAML

# TODO 5B-ii: Kubernetes Service YAML (LoadBalancer)
# Must contain:
#   apiVersion: v1, kind: Service
#   type: LoadBalancer
#   selector matching patient-risk-classifier
#   port: 80 → targetPort: 8000
K8S_SERVICE_YAML = None  # YOUR CODE — multi-line string YAML

# TODO 5B-iii: HorizontalPodAutoscaler (HPA) YAML
# Must contain:
#   apiVersion: autoscaling/v2, kind: HorizontalPodAutoscaler
#   scaleTargetRef pointing to patient-risk-classifier Deployment
#   minReplicas: 2, maxReplicas: 10
#   CPU target averageUtilization: 70
K8S_HPA_YAML = None  # YOUR CODE — multi-line string YAML

# TODO 5B-iv: Answer
Q_K8S_ROLLING = None  # What is a rolling update and why does it prevent downtime? (string)
Q_K8S_HPA     = None  # How does the HPA decide when to add or remove replicas? (string)

for name, content in [("deployment.yaml", K8S_DEPLOYMENT_YAML),
                       ("service.yaml",    K8S_SERVICE_YAML),
                       ("hpa.yaml",        K8S_HPA_YAML)]:
    if content:
        with open(os.path.join(BASE_DIR, name), "w") as f:
            f.write(content)
        print(f"--- {name} ---")
        print(content)

---
## Task 5C — CI/CD Pipeline Knowledge (4 points)

In [ ]:
# TODO 5C-i: Define 5 stages for an ML CI/CD pipeline
# Each stage: {'name': str, 'description': str}
CICD_PIPELINE_STAGES = [
    {"name": None, "description": None},  # e.g. code checkout + lint
    {"name": None, "description": None},  # e.g. unit tests
    {"name": None, "description": None},  # e.g. model train + evaluate
    {"name": None, "description": None},  # e.g. docker build + push
    {"name": None, "description": None},  # e.g. deploy to staging/prod
]

# TODO 5C-ii: Explain (string)
Q5_ROLLBACK  = None  # What is a rollback strategy?
Q5_CANARY    = None  # What is canary deployment? (in a CI/CD context)
Q5_BLUEGREEN = None  # What is blue-green deployment?

for i,s in enumerate(CICD_PIPELINE_STAGES,1):
    print(f"  {i}. {s['name']}: {s['description']}")

---
# Part 6 — Monitoring, Maintenance & MLOps (CLO6) — 25 Points

**CLO6**: Monitor and maintain deployed models.

| Task | Topic | Points |
|------|-------|--------|
| 6A | Performance Monitoring | 3 |
| 6B | Data Drift Detection | 3 |
| 6C | MLflow + WandB | 3 |
| 6D | Model Versioning | 3 |
| 6E | Retraining Pipeline | 3 |
| 6F | A/B Testing | 4 |
| 6G | Canary Deployment | 3 |
| 6H | Alerting & Incident Management | 3 |
| **Total** | | **25** |

---
## Task 6A — Model Performance Monitor (3 points)

In [ ]:
class ModelMonitor:
    """Tracks model performance in a production environment."""

    def __init__(self, model_name: str, accuracy_threshold: float = 0.80):
        self.model_name        = model_name
        self.accuracy_threshold = accuracy_threshold
        self.request_count     = 0
        self.correct_count     = 0
        self.latencies_ms      = []
        self.alerts            = []

    def log_prediction(self, predicted: str, actual: str, latency_ms: float):
        # TODO 6A-i:
        #   - Increment request_count
        #   - If predicted == actual: increment correct_count
        #   - Append latency_ms to latencies_ms
        #   - If request_count >= 10 and current_accuracy() < accuracy_threshold:
        #       append a string alert to self.alerts
        pass  # YOUR CODE

    def current_accuracy(self) -> float:
        # TODO 6A-ii: return correct_count / request_count, or 0.0 if no requests yet
        pass  # YOUR CODE

    def p95_latency(self) -> float:
        # TODO 6A-iii: return np.percentile(self.latencies_ms, 95), or 0.0 if empty
        pass  # YOUR CODE

    def summary(self) -> dict:
        return {
            "model_name":     self.model_name,
            "request_count":  self.request_count,
            "accuracy":       self.current_accuracy(),
            "p95_latency_ms": self.p95_latency(),
            "alert_count":    len(self.alerts),
        }


monitor = ModelMonitor("patient-risk-v1", accuracy_threshold=0.80)
for pred, true in zip(
    [CLASS_NAMES[p] for p in production_model.predict(X_test[:20])],
    [CLASS_NAMES[t] for t in y_test[:20]]
):
    monitor.log_prediction(pred, true, latency_ms=np.random.uniform(1, 20))

print("Monitor summary:", monitor.summary())

---
## Task 6B — Data Drift Detection (3 points)

We use the **Kolmogorov-Smirnov test**: if p-value < 0.05 the feature has drifted.

In [ ]:
# Drifted production data (age_norm + bp_norm have changed)
np.random.seed(99)
drifted_data = X_test.copy()
drifted_data[:, 0] += 2.0
drifted_data[:, 1] -= 1.5

# TODO 6B-i: detect_drift(reference, production, feature_names, alpha=0.05)
# For each feature: KS test on the corresponding columns
# Returns dict:
#   'drifted_features': list of drifted feature names
#   'feature_results' : {feature_name: {'p_value': float, 'drifted': bool}}
#   'overall_drift'   : bool
def detect_drift(reference, production, feature_names, alpha=0.05):
    pass  # YOUR CODE


# TODO 6B-ii: should_retrain(drift_result, max_drifted_features=2)
# True if len(drifted_features) >= max_drifted_features
def should_retrain(drift_result, max_drifted_features=2):
    pass  # YOUR CODE


drift_result   = detect_drift(X_train, drifted_data, FEATURE_NAMES)
retrain_needed = should_retrain(drift_result)

print("Drifted:", drift_result["drifted_features"])
print("Retrain:", retrain_needed)

---
## Task 6C — MLflow Experiment Tracking (3 points)

In [ ]:
import mlflow, mlflow.sklearn

mlflow.set_tracking_uri(f"file://{BASE_DIR}mlruns")
mlflow.set_experiment("patient-risk-classifier")

retrained_model = RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42)
retrained_model.fit(X_train, y_train)
retrain_accuracy = accuracy_score(y_test, retrained_model.predict(X_test))

# TODO 6C-i: Log an MLflow experiment named "retraining-run-v2"
# Log:
#   Params  : n_estimators=200, max_depth=10, random_state=42
#   Metrics : accuracy=retrain_accuracy, training_samples=len(X_train)
#   Tag     : model_type="RandomForest"
#   Model   : mlflow.sklearn.log_model(retrained_model, artifact_path="model")
# Store run_id in mlflow_run_id
mlflow_run_id = None  # YOUR CODE
# YOUR CODE: with mlflow.start_run(run_name="retraining-run-v2") as run: ...


# TODO 6C-ii: retraining strategy dict with keys:
#   'trigger_conditions' (list >= 3), 'retraining_frequency',
#   'validation_threshold' (float), 'rollback_on_degradation' (bool)
RETRAINING_STRATEGY = None  # YOUR CODE

# TODO 6C-iii: MLflow vs W&B comparison dict with keys:
#   'mlflow_best_for' : when to prefer MLflow (string, >10 chars)
#   'wandb_best_for'  : when to prefer W&B (string, >10 chars)
#   'key_difference'  : the main architectural/hosting difference (string, >10 chars)
TRACKING_TOOL_COMPARISON = None  # YOUR CODE

print(f"Retrained accuracy: {retrain_accuracy:.4f}")
print(f"MLflow run ID     : {mlflow_run_id}")

---
## Task 6D — Model Versioning Registry (3 points)

In MLOps, model version management is essential: track every version, promote the best one, and roll back on failure.

In [ ]:
class ModelVersionRegistry:
    """A simple registry for managing model versions."""

    def __init__(self):
        self.versions           = {}   # {version: {model, metrics, registered_at, stage}}
        self.production_version = None
        self._history           = []   # ordered list of promoted versions

    def register(self, model, version: str, metrics: dict):
        # TODO 6D-i: Add model to self.versions with stage='staging' and registered_at=datetime.utcnow().isoformat()
        # Raise ValueError if version already exists in self.versions
        pass  # YOUR CODE

    def promote(self, version: str):
        # TODO 6D-ii:
        #   1. Raise ValueError if version not in self.versions
        #   2. Set self.versions[version]['stage'] = 'production'
        #   3. Set self.production_version = version
        #   4. Append version to self._history
        pass  # YOUR CODE

    def rollback(self):
        # TODO 6D-iii:
        #   1. If production_version is None: return immediately
        #   2. Set self.versions[production_version]['stage'] = 'archived'
        #   3. Remove the last entry from self._history
        #   4. If self._history is not empty:
        #        prev = self._history[-1]
        #        self.versions[prev]['stage'] = 'production'
        #        self.production_version = prev
        #      Else:
        #        self.production_version = None
        pass  # YOUR CODE

    def get_production_model(self):
        # TODO 6D-iv: Return self.versions[self.production_version]['model'], or None if no production version
        pass  # YOUR CODE

    def list_versions(self) -> dict:
        return {
            v: {
                "stage":         info["stage"],
                "metrics":       info["metrics"],
                "registered_at": info["registered_at"],
            }
            for v, info in self.versions.items()
        }


# Test
registry = ModelVersionRegistry()
registry.register(production_model,  "v1.0.0", {"accuracy": test_accuracy})
registry.register(retrained_model,   "v2.0.0", {"accuracy": retrain_accuracy})
registry.promote("v1.0.0")
print("Production after promote v1:", registry.production_version)
registry.promote("v2.0.0")
print("Production after promote v2:", registry.production_version)
registry.rollback()
print("Production after rollback   :", registry.production_version)
print("Versions:", registry.list_versions())

---
## Task 6E — Automated Retraining Pipeline (3 points)

An automated pipeline that detects drift → retrains → evaluates → promotes or rolls back.

In [ ]:
# TODO 6E: Implement retraining_pipeline()
#
# Inputs:
#   X_train, y_train, X_test, y_test  — training and test data
#   drift_result   — result of detect_drift()
#   current_acc    — accuracy of the current model
#   registry       — ModelVersionRegistry
#   new_version    — name of the new version (string)
#   threshold=0.80 — minimum accuracy threshold
#
# Logic:
#   1. If should_retrain(drift_result) == False:
#        return {'retrained': False, 'reason': 'no_drift'}
#   2. Train a new model (RandomForestClassifier(n_estimators=200, random_state=0))
#   3. Compute new_acc on X_test
#   4. If new_acc >= threshold and new_acc >= current_acc:
#        register in registry + promote to production
#        return {'retrained': True, 'promoted': True, 'new_accuracy': new_acc, 'version': new_version}
#   5. Otherwise:
#        register in registry (stays in staging)
#        return {'retrained': True, 'promoted': False, 'new_accuracy': new_acc, 'reason': 'accuracy_degraded'}

def retraining_pipeline(X_train, y_train, X_test, y_test,
                        drift_result, current_acc, registry,
                        new_version, threshold=0.80):
    pass  # YOUR CODE


# Scenario 1: drifted data → retrain
registry2 = ModelVersionRegistry()
registry2.register(production_model, "v1.0.0", {"accuracy": test_accuracy})
registry2.promote("v1.0.0")

pipeline_result = retraining_pipeline(
    X_train, y_train, X_test, y_test,
    drift_result=drift_result,
    current_acc=test_accuracy,
    registry=registry2,
    new_version="v2.0.0"
)
print("Pipeline result (drift)    :", pipeline_result)

# Scenario 2: no drift → no retraining
no_drift_result = detect_drift(X_train, X_test, FEATURE_NAMES)
pipeline_result_no_drift = retraining_pipeline(
    X_train, y_train, X_test, y_test,
    drift_result=no_drift_result,
    current_acc=test_accuracy,
    registry=registry2,
    new_version="v3.0.0"
)
print("Pipeline result (no drift) :", pipeline_result_no_drift)

---
## Task 6F — A/B Testing: Compare Two Models in Production (4 points)

A/B Testing splits traffic between two models and compares their performance statistically.

In [ ]:
class ABTestRouter:
    """Routes requests between two models and tracks performance."""

    def __init__(self, model_a, model_b, traffic_to_b: float = 0.5):
        self.model_a      = model_a
        self.model_b      = model_b
        self.traffic_to_b = traffic_to_b
        self.results_a    = []   # list of 1 (correct) or 0 (wrong)
        self.results_b    = []

    def predict(self, features: list, true_label=None):
        # TODO 6F-i:
        #   1. Choose model: use B if np.random.random() < self.traffic_to_b, else A
        #   2. Run predict on np.array([features]) — returns a class index
        #   3. prediction = CLASS_NAMES[class_index]
        #   4. If true_label is not None: append 1/0 to the correct results list
        #   5. Return {'prediction': prediction, 'model': 'A' or 'B'}
        pass  # YOUR CODE

    def accuracy_a(self) -> float:
        return float(np.mean(self.results_a)) if self.results_a else 0.0

    def accuracy_b(self) -> float:
        return float(np.mean(self.results_b)) if self.results_b else 0.0

    def winner(self) -> str:
        # TODO 6F-ii: Return 'B' if accuracy_b() > accuracy_a() else 'A'
        # Return 'insufficient_data' if either results list is empty
        pass  # YOUR CODE

    def summary(self) -> dict:
        return {
            "requests_a": len(self.results_a),
            "requests_b": len(self.results_b),
            "accuracy_a": self.accuracy_a(),
            "accuracy_b": self.accuracy_b(),
            "winner":     self.winner(),
        }


# Test: production_model (A) vs retrained_model (B)
np.random.seed(7)
ab_router = ABTestRouter(production_model, retrained_model, traffic_to_b=0.5)
for i in range(200):
    ab_router.predict(
        features=X_test[i % len(X_test)].tolist(),
        true_label=y_test[i % len(y_test)]
    )

ab_summary = ab_router.summary()
print("A/B Summary:", ab_summary)

---
## Task 6G — Canary Deployment (3 points)

**Canary Deployment**: Start by routing a small fraction (10%) of traffic to the new model, then gradually increase it if performance is good, or roll back if it degrades.

In [ ]:
class CanaryDeployment:
    """Manages the gradual rollout of a new model."""

    def __init__(self, current_model, new_model, canary_fraction: float = 0.10):
        self.current_model   = current_model
        self.new_model       = new_model
        self.canary_fraction = canary_fraction
        self.current_metrics = []   # [1/0]
        self.canary_metrics  = []
        self.status          = "running"  # 'running' | 'promoted' | 'rolled_back'

    def route_request(self, features: list):
        # TODO 6G-i:
        #   1. use_canary = np.random.random() < self.canary_fraction
        #   2. Choose model accordingly and predict on np.array([features])
        #   3. prediction = CLASS_NAMES[class_index]
        #   4. Return {'prediction': prediction, 'model': 'canary' or 'current'}
        pass  # YOUR CODE

    def log_outcome(self, model_name: str, correct: int):
        # TODO 6G-ii: Append correct (1 or 0) to canary_metrics or current_metrics
        pass  # YOUR CODE

    def canary_accuracy(self) -> float:
        return float(np.mean(self.canary_metrics)) if self.canary_metrics else 0.0

    def current_accuracy(self) -> float:
        return float(np.mean(self.current_metrics)) if self.current_metrics else 0.0

    def increase_traffic(self, new_fraction: float):
        # TODO 6G-iii: Set self.canary_fraction = new_fraction
        # Raise ValueError if new_fraction > 1.0
        pass  # YOUR CODE

    def promote(self):
        self.canary_fraction = 1.0
        self.status = "promoted"

    def rollback(self):
        self.canary_fraction = 0.0
        self.status = "rolled_back"

    def summary(self) -> dict:
        return {
            "status":           self.status,
            "canary_fraction":  self.canary_fraction,
            "canary_accuracy":  self.canary_accuracy(),
            "current_accuracy": self.current_accuracy(),
            "canary_requests":  len(self.canary_metrics),
            "current_requests": len(self.current_metrics),
        }


np.random.seed(42)
canary = CanaryDeployment(production_model, retrained_model, canary_fraction=0.10)

# Phase 1: 10% canary
for i in range(100):
    feat = X_test[i % len(X_test)].tolist()
    res = canary.route_request(feat)
    true_cls = y_test[i % len(y_test)]
    pred_cls  = production_model.predict(np.array([feat]))[0] if res["model"]=="current" else retrained_model.predict(np.array([feat]))[0]
    canary.log_outcome(res["model"], int(pred_cls == true_cls))

print("Phase 1 (10%):", canary.summary())

# Phase 2: increase to 50%
canary.increase_traffic(0.50)
for i in range(100, 200):
    feat = X_test[i % len(X_test)].tolist()
    res = canary.route_request(feat)
    true_cls = y_test[i % len(y_test)]
    pred_cls  = production_model.predict(np.array([feat]))[0] if res["model"]=="current" else retrained_model.predict(np.array([feat]))[0]
    canary.log_outcome(res["model"], int(pred_cls == true_cls))

print("Phase 2 (50%):", canary.summary())

# Decision: if canary is better → promote, else → rollback
if canary.canary_accuracy() >= canary.current_accuracy():
    canary.promote()
else:
    canary.rollback()

print("Final status  :", canary.summary())

---
## Task 6H — Alerting & Incident Management (CLO6) — 3 points

Production ML services need automated alerts for four conditions:

| Check | Threshold | What it signals |
|-------|-----------|----------------|
| **Error rate** | > 1% of requests | Model or input validation is broken |
| **p99 latency** | > 500 ms | Model is too slow for SLO |
| **Mean confidence** | < 0.60 | Likely distribution shift |
| **Dead service** | 0 predictions in last 5 min | Service has crashed |

In [ ]:
from datetime import datetime as _dt, timedelta as _td

class AlertManager:
    """Checks production request logs against alert thresholds."""

    def check_error_rate(self, logs: list, threshold: float = 0.01):
        # TODO 6H-i:
        #   error_rate = count(log["error"] is True) / len(logs)
        #   Return a string alert if error_rate > threshold, else None
        if not logs:
            return None
        pass  # YOUR CODE

    def check_latency(self, logs: list, p99_threshold_ms: float = 500.0):
        # TODO 6H-ii:
        #   latencies = [r["latency_ms"] for r in logs if not r["error"]]
        #   p99 = np.percentile(latencies, 99)
        #   Return a string alert if p99 > p99_threshold_ms, else None
        latencies = [r["latency_ms"] for r in logs if not r.get("error", False)]
        if not latencies:
            return None
        pass  # YOUR CODE

    def check_confidence(self, logs: list, threshold: float = 0.6):
        # TODO 6H-iii:
        #   confidences = [r["confidence"] for r in logs if not r["error"]]
        #   mean_conf = np.mean(confidences)
        #   Return a string alert if mean_conf < threshold, else None
        confidences = [r["confidence"] for r in logs
                       if not r.get("error", False) and "confidence" in r]
        if not confidences:
            return None
        pass  # YOUR CODE

    def check_service_alive(self, logs: list, window_minutes: float = 5.0):
        # TODO 6H-iv:
        #   cutoff = datetime.utcnow() - timedelta(minutes=window_minutes)
        #   recent = [r for r in logs if not r["error"] and datetime.fromisoformat(r["timestamp"]) > cutoff]
        #   Return a string alert if recent is empty, else None
        cutoff = _dt.utcnow() - _td(minutes=window_minutes)
        pass  # YOUR CODE

    def run_all_checks(self, logs: list) -> list:
        alerts = []
        for check_fn in [self.check_error_rate, self.check_latency,
                         self.check_confidence, self.check_service_alive]:
            result = check_fn(logs)
            if result:
                alerts.append(result)
        return alerts


# ── Generate test logs with injected faults ───────────────────────────────────
_rng2 = np.random.default_rng(99)
_now2 = _dt.utcnow()
alert_logs = []
for i in range(200):
    _ts = _now2 - _td(minutes=10) + _td(seconds=i * 3)
    alert_logs.append({
        "timestamp": _ts.isoformat(),
        "latency_ms": round(float(_rng2.normal(40, 10)), 2),
        "confidence": round(float(np.clip(_rng2.normal(0.85, 0.08), 0.5, 1.0)), 4),
        "error": False,
    })

# Inject 15 errors (7.5% error rate — above 1% threshold)
for _idx in _rng2.choice(200, size=15, replace=False):
    alert_logs[_idx]["error"] = True
    alert_logs[_idx]["confidence"] = 0.0

# Inject 10 very slow requests
for _idx in _rng2.choice(200, size=10, replace=False):
    if not alert_logs[_idx]["error"]:
        alert_logs[_idx]["latency_ms"] = float(_rng2.uniform(700, 1200))

alert_manager = AlertManager()
fired_alerts = alert_manager.run_all_checks(alert_logs)

print(f"Alerts fired ({len(fired_alerts)}):")
for _a in fired_alerts:
    print(" ", _a)

---
# Closing Reflection — For Class Discussion

1. **CLO1**: What makes the AI model deployment lifecycle different from a traditional software development lifecycle?
2. **CLO2**: When would you choose ONNX over Pickle? In what situations is Pickle sufficient?
3. **CLO3**: What is the difference between REST and gRPC for model serving? When would you choose each?
4. **CLO4**: If you needed to serve one million requests per day, what cloud deployment strategy would you choose?
5. **CLO5**: What is the difference between a Container and a Virtual Machine? Why are containers better suited for ML?
6. **CLO6**: If an A/B test shows the new model outperforms the old one by only 0.1%, would you promote it? What factors influence your decision?

---
**Course References:**
- Machine Learning Engineering — Andriy Burkov (2020)
- Introducing MLOps — Treveil & Mané (2020)
- Practical MLOps — O'Reilly (2023)
- Engineering MLOps — Zhou (2023)